# Codecademy Final Project 1, part 2: Game Sale Analysis

In [6]:
#Previous notebook was running into memory errors from size
#splitting into two different notebooks for easier computational efficiency

#load files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

game_data = 'Video_Games.csv'
df = pd.read_csv(game_data)

#Clear data. data ends at 2016. cut off anything 2017+.
cutoff_date = 2016
df['Year_of_Release'] = pd.to_numeric(df['Year_of_Release'], errors='coerce').fillna(9999)
df_cleaned = df[df['Year_of_Release'] <= cutoff_date].copy()

print(f"Original: {df.shape[0]}")
print(f"After cleaning: {df_cleaned.shape[0]}")
print("=" * 70)

Original: 16719
After cleaning: 16446


5. How many years does it typically take for a new gaming platform to reach its peak sales? Which platforms had the longest/shortest lifecycles?

    Reason: For hardware manufacturers, understanding platform longevity informs R&D and marketing.

In [7]:
# ===== BLOCK 2: Aggregate Sales by Platform & Year =====
platform_year_sales = df_cleaned.groupby(['Platform', 'Year_of_Release'])['Global_Sales'].sum().reset_index()

In [8]:
first_game = df_cleaned.groupby('Platform')['Year_of_Release'].min().reset_index(name='First_Year')
first_game['First_Year'] = first_game['First_Year'].astype(int)

last_game = df_cleaned.groupby('Platform')['Year_of_Release'].max().reset_index(name='Last_Year')
last_game['Last_Year'] = last_game['Last_Year'].astype(int)

peak_info_list = []
for platform in df_cleaned['Platform'].unique():
    plat_df = df_cleaned[df_cleaned['Platform'] == platform].groupby('Year_of_Release')['Global_Sales'].sum()
    max_idx = int(plat_df.idxmax())
    max_sales = plat_df[max_idx]
    
    peak_info_list.append({
        'Platform': platform,
        'Peak_Year': max_idx,
        'Global_Sales': float(max_sales) / 1000000
    })

peak_info = pd.DataFrame(peak_info_list)

platform_summary = pd.merge(first_game, last_game, on='Platform', how='left')
platform_summary = platform_summary.merge(peak_info, on='Platform', how='left')

platform_summary['Lifecycle_Years'] = (platform_summary['Last_Year'] - platform_summary['First_Year'])
platform_summary['Years_to_Peak'] = (platform_summary['Peak_Year'] - platform_summary['First_Year'])

result_sorted = platform_summary.sort_values(['Global_Sales'], ascending=False).head(15)[['Platform', 'First_Year', 'Last_Year', 
                                                                                        'Lifecycle_Years', 'Peak_Year', 'Years_to_Peak']]

print("\n" + "=" * 70)
print("PLATFORM LONGEVITY ANALYSIS (Top 15 by Peak Sales)")
print("=" * 70)
print(f"\nPlatform              {'First':>8} {'Last':>8} {'LCYCLE':>10} {'PEAK':>8} {'T PEAK'}")
print("-" * 70)

for idx, row in result_sorted.iterrows():
    print("{} {:>8.0f} {:>8.0f} {:>10.1f} {:>8.0f} {:>12.1f}".format(
        str(row['Platform'])[:14],
        int(row['First_Year']),
        int(row['Last_Year']),
        row['Lifecycle_Years'],
        int(row['Peak_Year']),
        row['Years_to_Peak']))

print("\n" + "=" * 70)
print("KEY INSIGHTS:")
print("=" * 70)

longest = platform_summary.loc[platform_summary['Lifecycle_Years'].idxmax()]
shortest_no_na = platform_summary.sort_values('Lifecycle_Years').dropna().head(5)
fastest_peak = platform_summary.sort_values('Years_to_Peak', ascending=True).head(5)

print(f"\n🔹 LONGEST LIFECYCLE:")
print(f"   {longest['Platform']}: {int(longest['Lifecycle_Years'])} years ({int(longest['First_Year'])}-{int(longest['Last_Year'])})")

print("\n🔹 SHORTEST LIFECYCLES (by game release span):")
for _, r in shortest_no_na.iterrows():
    print(f"   {r['Platform']}: {str(int(r['Lifecycle_Years'])) if pd.notna(r['Lifecycle_Years']) else 'N/A'} years")

print("\n🔹 FASTEST TO REACH PEAK SALES:")
for _, r in fastest_peak.iterrows():
    print(f"   {r['Platform']}: peaked year {int(r['Peak_Year'])}, {int(abs(r['Years_to_Peak']))} years after launch")



PLATFORM LONGEVITY ANALYSIS (Top 15 by Peak Sales)

Platform                 First     Last     LCYCLE     PEAK T PEAK
----------------------------------------------------------------------
PS2     2000     2011       11.0     2004          4.0
Wii     2006     2016       10.0     2009          3.0
X360     2005     2016       11.0     2010          5.0
PS     1994     2003        9.0     1998          4.0
PS3     2006     2016       10.0     2011          5.0
DS     1985     2013       28.0     2007         22.0
PS4     2013     2016        3.0     2015          2.0
GBA     2000     2007        7.0     2004          4.0
XB     2000     2008        8.0     2004          4.0
GB     1988     2001       13.0     1989          1.0
3DS     2011     2016        5.0     2011          0.0
XOne     2013     2016        3.0     2015          2.0
N64     1996     2002        6.0     1999          3.0
PSP     2004     2015       11.0     2006          2.0
NES     1983     1994       11.0     1985

In [10]:
for platform in long_platforms['Platform']:
    df_plat = platform_year_sales[platform_year_sales['Platform'] == platform].sort_values('Year_of_Release')
    
    plt.figure(figsize=(10, 5))
    plt.plot(df_plat['Year_of_Release'], df_plat['Global_Sales'] / 1e6, marker='o', linewidth=2, color='#2E86AB')
    
    # Highlight peak
    peak_year = df_plat.loc[df_plat['Global_Sales'].idxmax(), 'Year_of_Release']
    peak_sales = df_plat['Global_Sales'].max() / 1e6
    plt.scatter(peak_year, peak_sales, color='#E63946', s=200, zorder=5, label=f'Peak: {peak_year}')
    
    plt.title(f'{platform} – Global Sales Over Time', fontsize=14)
    plt.xlabel('Year')
    plt.ylabel('Global Sales (Millions)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

NameError: name 'long_platforms' is not defined

#### ANALYSIS:
One major thing I notice in these array of graphs is the longevity of the PC market. The PC market stems unnaturally long compared to the other consoles on this list, stemming from the late 1980s to the present. Most consoles (being more dated) die out before the end of the study's scope. This long track record of PC Gaming illustrates the adaptability and resilience of the PC market, with dedicated PC gamers adapting to the shifting technologies of the time.

Comparatively, the SEGA Dreamcast (DC) had a very short run, from 1998 to 2002, only 4 years. This extremely brief lifecycle has to do with the downsizing of SEGA during this period in the gaming industry (https://en.wikipedia.org/wiki/Dreamcast#Decline). The Dreamcast fell short on the initial hopes for its sales figures, and internal restrucutring and conflicts of opinion with the higher-ups in SEGA resulted in its brief performance in the market, despite the high praises and strong game library in its own right. Sega transitioned into game publishing and software full time after the Dreamcast was discontinued.

Finally, we can examine the sales figures of the NES, and identify two peaks (https://vgsales.fandom.com/wiki/Nintendo_Entertainment_System). one in 1985, and another in 1988, and a smaller one in 1990. This has to do with the various high-performing games that were released during these years alongside the release dates of the main console in all 3 main regions, Japan, US, and EU, in staggering years. The peak in 1985 coincides with the release of the NES in the US. 1988 peak coincides with some of the highest selling games in the NES catalogue with Super Mario Bros 2 and 3. reasons for the final small peak in 1990 are unclear.

# 6. Which publishers are most dominant in each genre? (“Genre specialists” vs. “generalists”)

    Reason: Reveals strategic positioning – e.g., Nintendo owns Platform/Sports, Take-Two dominates Action.

In [11]:
# Clean data - remove rows with missing values in key columns
df_cleaned = df.dropna(subset=['Global_Sales', 'Genre', 'Publisher']).copy()

print("Dataset shape:", df.shape)
print(f"After cleaning: {df_cleaned.shape[0]} records")

Dataset shape: (16719, 16)
After cleaning: 16663 records


# 7. What is the relationship between a game’s Rating (ESRB: E, E10+, T, M) and its sales in different regions? Does M-rated games sell better in NA vs. JP?

    Reason: Cultural differences in acceptance of mature content affect localization and marketing.

# 8. Are there “hidden gems” – games with high critic scores but very low global sales? Which developers create consistently high-quality niche titles?

    Reason: For investors or publishers looking for acquisition targets or underserved markets.